<a href="https://colab.research.google.com/github/Shekinah-Ineza/formative3-group19/blob/main/Formative3_Part2_Bayes_Team19.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Formative 3, Part 2: Bayesian Probability on IMDb Reviews Team 19

  |Teammates: MUTONI KEIRA  Habibllah Ayoola Ayodele Merveille Shekina Ineza
  
We apply Bayes' theorem to estimate how likely a review is positive when it contains a given keyword, using the IMDb 50K movie reviews dataset and basic Python only.

We chose Excellent, Wonderful and Amazing for positive words. And we chose Boring, Terrible and Waste for negative words.

### Why we chose these keywords

| Keyword | Expected | Why we chose it |
|---|---|---|
| excellent | Positive | A strong, direct word of praise that is rarely used sarcastically. |
| wonderful | Positive | Expresses enjoyment and emotion, common in reviews of films people liked. |
| amazing | Positive | Everyday word for strong approval, very common in casual reviews. |
| terrible | Negative | A strong, direct word of dislike. |
| waste | Negative | Used in "waste of time / money", a classic complaint about a film. |
| boring | Negative | The most common reason people give for disliking a film. |

In [1]:
import csv
import glob
import os

csv.field_size_limit(10_000_000)   # some reviews are very long

def find_dataset():
    """Find the IMDb CSV next to this notebook, in a part2 folder, or in Colab."""
    candidates = ["IMDB Dataset.csv", "/content/IMDB Dataset.csv"] + glob.glob("**/IMDB*.csv", recursive=True)
    for path in candidates:
        if os.path.exists(path):
            return path
    raise FileNotFoundError("IMDB Dataset.csv not found. Download it from Kaggle and put it next to this notebook.")

def load_reviews(path):
    """Read the CSV and return (review_text, sentiment) pairs.
    Labels are cleaned, and empty or unknown rows are skipped and counted."""
    reviews, skipped = [], 0
    with open(path, encoding="utf-8", newline="") as f:
        for row in csv.DictReader(f):
            text = (row.get("review") or "").strip()
            label = (row.get("sentiment") or "").strip().lower()
            if not text or label not in ("positive", "negative"):
                skipped += 1
                continue
            reviews.append((text, label))
    return reviews, skipped

DATA_PATH = find_dataset()
reviews, skipped = load_reviews(DATA_PATH)
print("Using:", DATA_PATH)
print("Total reviews:", len(reviews), "| skipped rows:", skipped)

Using: IMDB Dataset.csv
Total reviews: 50000 | skipped rows: 0


In [2]:
import re

HTML_TAG = re.compile(r"<[^>]+>")              # removes <br /> and any other tag
WORD = re.compile(r"[a-z]+(?:'[a-z]+)?")       # words only; keeps "don't" as one word

def tokenize(text):
    """Lowercase, remove HTML tags, numbers and all punctuation, return the set of words.
    Using a set means a word counts once per review."""
    text = HTML_TAG.sub(" ", text.lower())
    return set(WORD.findall(text))

def prepare(reviews):
    """Return a list of (word_set, is_positive) pairs."""
    return [(tokenize(text), label == "positive") for text, label in reviews]

data = prepare(reviews)
print("Prepared:", len(data))
print(sorted(tokenize("GREAT film!!<br /><br />I don't regret it... (excellent)")))

Prepared: 50000
["don't", 'excellent', 'film', 'great', 'i', 'it', 'regret']


In [3]:
def bayes_for_keyword(data, keyword):
    """Compute prior, likelihood, marginal and posterior P(Positive | keyword)."""
    total = len(data)
    n_pos = sum(1 for words, is_pos in data if is_pos)
    n_keyword = sum(1 for words, is_pos in data if keyword in words)
    n_keyword_and_pos = sum(1 for words, is_pos in data if is_pos and keyword in words)

    if n_keyword == 0:
        return None  # keyword never appears

    prior = n_pos / total                      # P(Positive)
    likelihood = n_keyword_and_pos / n_pos     # P(keyword | Positive)
    marginal = n_keyword / total               # P(keyword)
    posterior = likelihood * prior / marginal  # P(Positive | keyword)

    return {"keyword": keyword, "prior": prior, "likelihood": likelihood,
            "marginal": marginal, "posterior": posterior,
            "check": n_keyword_and_pos / n_keyword}

In [4]:
positive_keywords = ["excellent", "wonderful", "amazing"]
negative_keywords = ["terrible", "waste", "boring"]

print(f"{'keyword':<12}{'prior':>8}{'likelihood':>12}{'marginal':>10}{'posterior':>11}{'check':>8}")
for kw in positive_keywords + negative_keywords:
    r = bayes_for_keyword(data, kw)
    if r is None:
        print(f"{kw:<12}  not found in any review (posterior undefined)")
    else:
        print(f"{kw:<12}{r['prior']:>8.4f}{r['likelihood']:>12.4f}{r['marginal']:>10.4f}{r['posterior']:>11.4f}{r['check']:>8.4f}")

keyword        prior  likelihood  marginal  posterior   check
excellent     0.5000      0.1147    0.0710     0.8074  0.8074
wonderful     0.5000      0.0902    0.0556     0.8121  0.8121
amazing       0.5000      0.0671    0.0431     0.7779  0.7779
terrible      0.5000      0.0153    0.0540     0.1415  0.1415
waste         0.5000      0.0070    0.0507     0.0687  0.0687
boring        0.5000      0.0237    0.0610     0.1940  0.1940


In [5]:
def show_keyword_block(keyword, result):
    """Print one small table for a single keyword, or explain why it can't be computed."""
    if result is None:
        print(f"Keyword: {keyword}")
        print("  Not found in any review, so P(keyword) = 0 and Bayes' theorem would divide by zero.")
        print("  The posterior is undefined for this keyword.\n")
        return
    print(f"Keyword: {keyword}")
    print(f"  Prior      P(Positive)           = {result['prior']:.4f}")
    print(f"  Likelihood P(keyword | Positive) = {result['likelihood']:.4f}")
    print(f"  Marginal   P(keyword)            = {result['marginal']:.4f}")
    print(f"  Posterior  P(Positive | keyword) = {result['posterior']:.4f}")
    print()

for kw in positive_keywords + negative_keywords:
    show_keyword_block(kw, bayes_for_keyword(data, kw))

# Absent-keyword check: a made-up word that appears in no review
show_keyword_block("zzzznotaword", bayes_for_keyword(data, "zzzznotaword"))

Keyword: excellent
  Prior      P(Positive)           = 0.5000
  Likelihood P(keyword | Positive) = 0.1147
  Marginal   P(keyword)            = 0.0710
  Posterior  P(Positive | keyword) = 0.8074

Keyword: wonderful
  Prior      P(Positive)           = 0.5000
  Likelihood P(keyword | Positive) = 0.0902
  Marginal   P(keyword)            = 0.0556
  Posterior  P(Positive | keyword) = 0.8121

Keyword: amazing
  Prior      P(Positive)           = 0.5000
  Likelihood P(keyword | Positive) = 0.0671
  Marginal   P(keyword)            = 0.0431
  Posterior  P(Positive | keyword) = 0.7779

Keyword: terrible
  Prior      P(Positive)           = 0.5000
  Likelihood P(keyword | Positive) = 0.0153
  Marginal   P(keyword)            = 0.0540
  Posterior  P(Positive | keyword) = 0.1415

Keyword: waste
  Prior      P(Positive)           = 0.5000
  Likelihood P(keyword | Positive) = 0.0070
  Marginal   P(keyword)            = 0.0507
  Posterior  P(Positive | keyword) = 0.0687

Keyword: boring
  Prior    

In [6]:
def show_keyword_counts(data, keywords):
    """Display the review counts behind each probability."""
    total = len(data)
    positive = sum(is_positive for _, is_positive in data)

    print(f"Total reviews: {total}")
    print(f"Positive reviews: {positive}\n")
    print(f"{'Keyword':<12} {'Contains':>10} {'Positive + word':>17}")

    for keyword in keywords:
        contains = sum(
            keyword in words
            for words, _ in data
        )
        positive_contains = sum(
            is_positive and keyword in words
            for words, is_positive in data
        )

        print(
            f"{keyword:<12} {contains:>10} "
            f"{positive_contains:>17}"
        )


keywords = positive_keywords + negative_keywords
show_keyword_counts(data, keywords)

Total reviews: 50000
Positive reviews: 25000

Keyword        Contains   Positive + word
excellent          3552              2868
wonderful          2778              2256
amazing            2157              1678
terrible           2699               382
waste              2533               174
boring             3052               592


In [7]:
def show_bayes_example(data, keyword):
    """Show substitution into Bayes' theorem."""
    result = bayes_for_keyword(data, keyword)

    if result is None:
        print(f"No reviews contain '{keyword}'.")
        return

    prior = result["prior"]
    likelihood = result["likelihood"]
    marginal = result["marginal"]
    posterior = result["posterior"]

    print(f"Keyword: {keyword}")
    print("P(Positive | word)")
    print("= P(word | Positive) × P(Positive) / P(word)")
    print(f"= {likelihood:.6f} × {prior:.6f} / {marginal:.6f}")
    print(f"= {posterior:.6f} ({posterior:.2%})")


show_bayes_example(data, "excellent")

Keyword: excellent
P(Positive | word)
= P(word | Positive) × P(Positive) / P(word)
= 0.114720 × 0.500000 / 0.071040
= 0.807432 (80.74%)


### Understanding the counts

A keyword is counted once per review, regardless of how often
it appears in that review.

The prior uses all reviews. The likelihood uses positive reviews.
The marginal measures how frequently the keyword appears across
all reviews. Bayes' theorem combines these to calculate the
probability that a review is positive given that keyword.

This measures an association in this dataset. A keyword alone
does not guarantee the sentiment of an individual review.

In [8]:
def interpret(keyword, result):
    """One plain-English sentence comparing the posterior to the prior."""
    if result is None:
        return f"'{keyword}' never appears, so it tells us nothing about sentiment."
    post, prior = result["posterior"], result["prior"]
    lean = "positive" if post > prior else "negative"
    return (f"A review containing '{keyword}' is {post:.1%} likely to be positive "
            f"(vs {prior:.0%} for any review), so the word leans {lean}.")

for kw in positive_keywords + negative_keywords:
    print("-", interpret(kw, bayes_for_keyword(data, kw)))

- A review containing 'excellent' is 80.7% likely to be positive (vs 50% for any review), so the word leans positive.
- A review containing 'wonderful' is 81.2% likely to be positive (vs 50% for any review), so the word leans positive.
- A review containing 'amazing' is 77.8% likely to be positive (vs 50% for any review), so the word leans positive.
- A review containing 'terrible' is 14.2% likely to be positive (vs 50% for any review), so the word leans negative.
- A review containing 'waste' is 6.9% likely to be positive (vs 50% for any review), so the word leans negative.
- A review containing 'boring' is 19.4% likely to be positive (vs 50% for any review), so the word leans negative.


### Interpretation and conclusion

The dataset is balanced (25,000 positive and 25,000 negative reviews), so the prior is 0.5. Any keyword that pushes the posterior above 50% points to a positive review, and below 50% to a negative one.

- **Positive keywords:** "wonderful" and "excellent" both give about 81% positive, and "amazing" about 78%. All three are clear positive signals.
- **Negative keywords:** "waste" is the strongest signal of all: only about 7% of reviews containing it are positive, so about 93% are negative. "terrible" (about 14% positive) and "boring" (about 19%) are also strong negative signs.
- **Negative words were stronger than positive ones.** People rarely write "waste" or "terrible" in a review they liked, but positive words do appear in mixed or negative reviews ("excellent cast, but a boring plot").
- **Limits:** Bayes here only checks whether a word appears, not its context, so "not boring" still counts as "boring". A keyword is a strong hint, never a guarantee.

The "check" column (positive reviews with the word divided by all reviews with the word) matches the Bayes posterior exactly, which confirms our implementation.